#1. Web Scrapping

## import library

In [1]:
!pip install requests beautifulsoup4 pandas tqdm lxml -q

In [2]:
import requests
import pandas as pd
import time
import random

from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

##configuration

In [3]:
REQUEST_TIMEOUT = 15

MIN_DELAY = 0.5
MAX_DELAY = 1.0
MAX_PAGE = 10

SAVE_FOLDER = "dataset"

USER_AGENT = {
    "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0 Safari/537.36"
}

##http session

In [4]:
retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429,500,502,503,504]
)

adapter = HTTPAdapter(max_retries=retry_strategy)

http = requests.Session()

http.mount("https://", adapter)
http.mount("http://", adapter)

http.headers.update(USER_AGENT)

##function untuk download halaman

In [5]:
def download_page(url):

    try:

        response = http.get(
            url,
            timeout=REQUEST_TIMEOUT
        )

        response.raise_for_status()

        return response.text

    except Exception:

        return None

##random delay

In [6]:
def random_delay():

    time.sleep(
        random.uniform(
            MIN_DELAY,
            MAX_DELAY
        )
    )

##cek koneksi

In [7]:
test_url = "https://sport.detik.com/sepakbola/liga-inggris"

html = download_page(test_url)

if html:
    print("Berhasil mengambil halaman.")
    print(html[:500])
else:
    print("Gagal mengambil halaman.")

Berhasil mengambil halaman.
<!DOCTYPE html>
<html lang="id-ID">
    <head>
        <script type="text/javascript" src="https://iat.detiknetwork.com/ip-information.js"></script>
        <link rel="preconnect" href="https://awscdn.detik.net.id"><link rel="preconnect" href="https://awscdn.detik.net.id" crossorigin><link rel="preconnect" href="https://cdn.detik.net.id"><link rel="preconnect" href="https://cdn.detik.net.id" crossorigin>        <link rel="dns-prefetch" href="https://cdn.detik.net.id"/><link rel="dns-prefetch" h


##kategori detik.com

In [8]:
detik_categories = {

    "Liga Inggris": {
        "url": "https://sport.detik.com/sepakbola/liga-inggris",
        "label": "Liga Inggris",
        "target": 100
    },

    "Liga Italia": {
        "url": "https://sport.detik.com/sepakbola/liga-italia",
        "label": "Liga Italia",
        "target": 100
    },

    "Liga Spanyol": {
        "url": "https://sport.detik.com/sepakbola/liga-spanyol",
        "label": "Liga Spanyol",
        "target": 100
    },

    "Liga Indonesia": {
        "url": "https://sport.detik.com/sepakbola/liga-indonesia",
        "label": "Liga Indonesia",
        "target": 100
    },

    "F1": {
        "url": "https://sport.detik.com/f1",
        "label": "Non Sepak Bola",
        "target": 100
    },

    "MotoGP": {
        "url": "https://sport.detik.com/moto-gp",
        "label": "Non Sepak Bola",
        "target": 100
    },

    "Badminton": {
        "url": "https://sport.detik.com/raket",
        "label": "Non Sepak Bola",
        "target": 100
    }

}

##fungsi index url

In [9]:
def create_index_url(base_url, date, page):

    return (
        f"{base_url}/indeks"
        f"?date={date}"
        f"&page={page}"
    )

##ambil link artikel

In [10]:
def extract_links(html):

    soup = BeautifulSoup(html, "html.parser")

    article_links = []

    articles = soup.select("article.list-content__item")

    for item in articles:

        link = item.find("a", href=True)

        if link:

            url = link["href"]

            if url.startswith("https://sport.detik.com"):

                article_links.append(url)

    return article_links

In [11]:
def collect_detik_urls(base_url, total_needed):

    collected = []

    visited = set()

    today = datetime.today()

    for day in range(120):

        current_day = today - timedelta(days=day)

        date_string = current_day.strftime("%m/%d/%Y")

        for page in range(1,11):

            url = create_index_url(
                base_url,
                date_string,
                page
            )

            html = download_page(url)

            if html is None:
                continue

            links = extract_links(html)

            if len(links)==0:
                break

            for item in links:

                if item not in visited:

                    visited.add(item)

                    collected.append(item)

            print(
                f"{date_string} | Page {page} | Total : {len(collected)}"
            )

            if len(collected)>=total_needed:

                return collected[:total_needed]

            random_delay()

    return collected

In [12]:
urls = collect_detik_urls(

    detik_categories["Liga Inggris"]["url"],

    20

)

print()

print("Jumlah URL :",len(urls))

print()

for i in urls[:10]:

    print(i)

06/29/2026 | Page 1 | Total : 7
06/28/2026 | Page 1 | Total : 9
06/27/2026 | Page 1 | Total : 13
06/26/2026 | Page 1 | Total : 17
06/25/2026 | Page 1 | Total : 19
06/24/2026 | Page 1 | Total : 22

Jumlah URL : 20

https://sport.detik.com/sepakbola/liga-inggris/d-8552996/terungkap-enzo-maresca-sudah-incar-posisi-pep-sejak-masih-di-chelsea
https://sport.detik.com/sepakbola/liga-inggris/d-8552949/bukan-maksud-maresca-untuk-hancurkan-chelsea
https://sport.detik.com/sepakbola/liga-inggris/d-8552923/city-akui-dekati-maresca-sejak-masih-di-chelsea-setuju-bayar-kompensasi
https://sport.detik.com/sepakbola/liga-inggris/d-8552883/enzo-maresca-resmi-menjadi-manajer-manchester-city
https://sport.detik.com/sepakbola/liga-inggris/d-8551977/liverpool-jeblok-musim-lalu-klopp-enggan-menghakimi
https://sport.detik.com/sepakbola/liga-inggris/d-8551633/diomande-lebih-pilih-psg-liverpool-gigit-jari
https://sport.detik.com/sepakbola/liga-inggris/d-8551462/chelsea-sudah-tawar-granit-xhaka-ditolak-mentah-ment

##function get judul artikel

In [13]:
def get_title(soup):

    title = soup.find("title")

    if title:

        return title.get_text(strip=True)

    return ""

In [14]:
def clean_article(article):

    remove_tags = [
        "script",
        "style",
        "table",
        "figure",
        "aside",
        "iframe",
        "noscript",
        "ins"
    ]

    for tag in remove_tags:

        for item in article.find_all(tag):

            item.decompose()

    return article

In [15]:
def get_article_text(soup):

    content = soup.find(
        "div",
        class_="detail__body-text"
    )

    if content is None:

        return None

    content = clean_article(content)

    paragraphs = content.find_all("p")

    article = []

    for p in paragraphs:

        text = p.get_text(
            separator=" ",
            strip=True
        )

        if len(text):

            article.append(text)

    return " ".join(article)

In [16]:
def scrape_article(url):

    html = download_page(url)

    if html is None:

        return None

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    title = get_title(soup)

    body = get_article_text(soup)

    if body is None:

        return None

    return {

        "title": title,

        "text": body

    }

In [17]:
sample = urls[0]

data = scrape_article(sample)

print(data["title"])

print()

print(data["text"][:1000])

Terungkap! Enzo Maresca Sudah Incar Posisi Pep Sejak Masih di Chelsea

Enzo Maresca resmi menjadi penerus Pep Guardiola di Manchester City . Dia rupanya sudah mengincar kursi pelatih The Sky Blues sejak masih membesut Chelsea . Man City mengumumkan penunjukan Maresca sebagai manajer baru tim pada Senin (29/6/2026) malam WIB. Juru taktik asal Italia itu diikat kontrak selama 3 tahun di Etihad Stadium. Maresca menjadi suksesor Guardiola yang telah menangani Man City selama 10 tahun. Sebelum menerima tawaran The Citizens , Maresca sempat melatih Chelsea pada 2024 hingga 2026. SCROLL TO CONTINUE WITH CONTENT Di Chelsea Maresca sukses mempersembahkan titel Piala Dunia Antarklub 2025. Keputusan mengejutkan diambilnya pada awal tahun 2026 dengan mengundurkan diri dari jabatan manajer The Blues. Beberapa laporan menyebut Maresca cabut dari Chelsea karena berselisih dengan direksi klub. London Biru justru berdalih pelatih berusia 36 tahun itu meninggalkan Stamford Bridge karena sudah yakin baka

In [18]:
detik_urls = []

for category_name, config in detik_categories.items():

    print("="*70)
    print(f"Category : {category_name}")

    urls = collect_detik_urls(
        base_url=config["url"],
        total_needed=config["target"]
    )

    print(f"Collected : {len(urls)}")

    for link in urls:

        detik_urls.append({

            "url": link,

            "label": config["label"],

            "subcategory": category_name,

            "media": "Detik"

        })

    print()

Category : Liga Inggris
06/29/2026 | Page 1 | Total : 7
06/28/2026 | Page 1 | Total : 9
06/27/2026 | Page 1 | Total : 13
06/26/2026 | Page 1 | Total : 17
06/25/2026 | Page 1 | Total : 19
06/24/2026 | Page 1 | Total : 22
06/23/2026 | Page 1 | Total : 24
06/21/2026 | Page 1 | Total : 25
06/20/2026 | Page 1 | Total : 29
06/19/2026 | Page 1 | Total : 34
06/18/2026 | Page 1 | Total : 35
06/17/2026 | Page 1 | Total : 37
06/16/2026 | Page 1 | Total : 38
06/15/2026 | Page 1 | Total : 39
06/13/2026 | Page 1 | Total : 40
06/12/2026 | Page 1 | Total : 42
06/11/2026 | Page 1 | Total : 47
06/10/2026 | Page 1 | Total : 50
06/09/2026 | Page 1 | Total : 52
06/08/2026 | Page 1 | Total : 56
06/06/2026 | Page 1 | Total : 63
06/05/2026 | Page 1 | Total : 78
06/04/2026 | Page 1 | Total : 84
06/03/2026 | Page 1 | Total : 94
06/02/2026 | Page 1 | Total : 103
Collected : 100

Category : Liga Italia
06/28/2026 | Page 1 | Total : 1
06/27/2026 | Page 1 | Total : 2
06/25/2026 | Page 1 | Total : 3
06/24/2026 | Pag

In [19]:
detik_df = pd.DataFrame(detik_urls)

print(detik_df.shape)

detik_df.head()

(622, 4)


,url,label,subcategory,media
0,https://sport.detik.com/sepakbola/liga-inggris...,Liga Inggris,Liga Inggris,Detik
1,https://sport.detik.com/sepakbola/liga-inggris...,Liga Inggris,Liga Inggris,Detik
2,https://sport.detik.com/sepakbola/liga-inggris...,Liga Inggris,Liga Inggris,Detik
3,https://sport.detik.com/sepakbola/liga-inggris...,Liga Inggris,Liga Inggris,Detik
4,https://sport.detik.com/sepakbola/liga-inggris...,Liga Inggris,Liga Inggris,Detik


In [20]:
detik_df = detik_df.drop_duplicates(
    subset="url"
).reset_index(drop=True)

print(detik_df.shape)

(622, 4)


In [21]:
detik_df.groupby(
    ["label","subcategory"]
).size()

label           subcategory   
Liga Indonesia  Liga Indonesia    100
Liga Inggris    Liga Inggris      100
Liga Italia     Liga Italia       100
Liga Spanyol    Liga Spanyol      100
Non Sepak Bola  Badminton         100
                F1                 22
                MotoGP            100
dtype: int64

##kategori Antara

In [22]:
antara_categories = {

    "Liga Inggris": {
        "url": "https://www.antaranews.com/tag/liga-inggris",
        "label": "Liga Inggris",
        "target": 100
    },

    "Liga Italia": {
        "url": "https://www.antaranews.com/tag/liga-italia",
        "label": "Liga Italia",
        "target": 100
    },

    "Liga Spanyol": {
        "url": "https://www.antaranews.com/tag/liga-spanyol",
        "label": "Liga Spanyol",
        "target": 100
    },

    "Liga Indonesia": {
        "url": "https://www.antaranews.com/tag/liga-indonesia",
        "label": "Liga Indonesia",
        "target": 100
    },

    "Basket": {
        "url": "https://www.antaranews.com/tag/basket",
        "label": "Non Sepak Bola",
        "target": 100
    },

    "Bulutangkis": {
        "url": "https://www.antaranews.com/tag/bulutangkis",
        "label": "Non Sepak Bola",
        "target": 100
    }

}

In [23]:
def build_tag_page(base_url, page):

    if page == 1:
        return base_url

    return f"{base_url}/{page}"

In [25]:
def parse_article_links(html):

    soup = BeautifulSoup(html, "html.parser")

    links = []
    visited = set()

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/berita/" not in href:
            continue

        if not href.startswith("https://www.antaranews.com"):
            continue

        if href in visited:
            continue

        visited.add(href)

        links.append(href)

    return links

In [26]:
def collect_antara_urls(base_url, target):

    article_urls = []

    unique_urls = set()

    page = 1

    while len(article_urls) < target:

        page_url = build_tag_page(base_url, page)

        html = download_page(page_url)

        if html is None:
            break

        links = parse_article_links(html)

        if len(links) == 0:
            break

        for url in links:

            if url not in unique_urls:

                unique_urls.add(url)

                article_urls.append(url)

        print(f"Page {page} | Total : {len(article_urls)}")

        page += 1

        random_delay()

    return article_urls[:target]

In [27]:
urls = collect_antara_urls(

    antara_categories["Liga Inggris"]["url"],

    20

)

print()

print(len(urls))

print()

for i in urls[:10]:

    print(i)

Page 1 | Total : 35

20

https://www.antaranews.com/berita/5628093/chelsea-maresca-sudah-berniat-latih-city-meski-masih-punya-kontrak
https://www.antaranews.com/berita/5627856/enzo-maresca-resmi-latih-manchester-city
https://www.antaranews.com/berita/5626679/brasil-vs-jepang-bentrok-dua-ambisi-untuk-hasilkan-platform-juara
https://www.antaranews.com/berita/5626077/pelatih-bangga-kongo-ukir-sejarah-usai-lolos-ke-32-besar-piala-dunia
https://www.antaranews.com/berita/5623528/norwegia-vs-prancis-duel-dua-calon-peraih-sepatu-emas
https://www.antaranews.com/berita/5623525/manchester-city-dilaporkan-capai-kesepakatan-rekrut-elliot-anderson
https://www.antaranews.com/berita/5623491/arsenal-resmi-permanenkan-bek-piero-hincapie-dari-bayer-leverkusen
https://www.antaranews.com/berita/5621664/visa-ditolak-karier-menanjak-kisah-bek-andalan-kolombia-daniel-munoz
https://www.antaranews.com/berita/5620195/dominasi-messi-alarm-bahaya-argentina-di-piala-dunia
https://www.antaranews.com/berita/5619055/h

In [28]:
antara_urls = []

for category_name, config in antara_categories.items():

    print("=" * 70)
    print(f"Category : {category_name}")

    urls = collect_antara_urls(
        base_url=config["url"],
        target=config["target"]
    )

    print(f"Collected : {len(urls)}")

    for url in urls:

        antara_urls.append({

            "url": url,
            "label": config["label"],
            "subcategory": category_name,
            "media": "Antara"

        })

    print()

Category : Liga Inggris
Page 1 | Total : 35
Page 2 | Total : 50
Page 3 | Total : 65
Page 4 | Total : 80
Page 5 | Total : 101
Collected : 100

Category : Liga Italia
Page 1 | Total : 35
Page 2 | Total : 50
Page 3 | Total : 65
Page 4 | Total : 77
Page 5 | Total : 92
Page 6 | Total : 105
Collected : 100

Category : Liga Spanyol
Page 1 | Total : 35
Page 2 | Total : 50
Page 3 | Total : 65
Page 4 | Total : 80
Page 5 | Total : 94
Page 6 | Total : 109
Collected : 100

Category : Liga Indonesia
Page 1 | Total : 34
Page 2 | Total : 42
Page 3 | Total : 56
Page 4 | Total : 68
Page 5 | Total : 79
Page 6 | Total : 91
Page 7 | Total : 101
Collected : 100

Category : Basket
Page 1 | Total : 34
Page 2 | Total : 49
Page 3 | Total : 63
Page 4 | Total : 77
Page 5 | Total : 91
Page 6 | Total : 106
Collected : 100

Category : Bulutangkis
Page 1 | Total : 27
Page 2 | Total : 37
Page 3 | Total : 47
Page 4 | Total : 56
Page 5 | Total : 68
Page 6 | Total : 82
Page 7 | Total : 97
Page 8 | Total : 108
Collected :

In [29]:
antara_df = pd.DataFrame(antara_urls)

print(antara_df.shape)

antara_df.head()

(600, 4)


,url,label,subcategory,media
0,https://www.antaranews.com/berita/5628093/chel...,Liga Inggris,Liga Inggris,Antara
1,https://www.antaranews.com/berita/5627856/enzo...,Liga Inggris,Liga Inggris,Antara
2,https://www.antaranews.com/berita/5626679/bras...,Liga Inggris,Liga Inggris,Antara
3,https://www.antaranews.com/berita/5626077/pela...,Liga Inggris,Liga Inggris,Antara
4,https://www.antaranews.com/berita/5623528/norw...,Liga Inggris,Liga Inggris,Antara


In [30]:
antara_df.groupby(
    ["label", "subcategory"]
).size()

label           subcategory   
Liga Indonesia  Liga Indonesia    100
Liga Inggris    Liga Inggris      100
Liga Italia     Liga Italia       100
Liga Spanyol    Liga Spanyol      100
Non Sepak Bola  Basket            100
                Bulutangkis       100
dtype: int64

In [31]:

all_links = pd.concat(
    [detik_df, antara_df],
    ignore_index=True
)

all_links = (
    all_links
    .drop_duplicates(subset="url")
    .reset_index(drop=True)
)

print(all_links.shape)

all_links.groupby(
    ["media", "label"]
).size()

(1104, 4)


media   label         
Antara  Liga Indonesia     79
        Liga Inggris      100
        Liga Italia        74
        Liga Spanyol       69
        Non Sepak Bola    160
Detik   Liga Indonesia    100
        Liga Inggris      100
        Liga Italia       100
        Liga Spanyol      100
        Non Sepak Bola    222
dtype: int64

##detik scrapper

In [32]:
def scrape_detik(html):

    soup = BeautifulSoup(html, "html.parser")

    content = soup.find("div", class_="detail__body-text")

    if content is None:
        return None

    for tag in content(["script","style","table","figure","aside","iframe","ins"]):
        tag.decompose()

    paragraphs = content.find_all("p")

    text = " ".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if p.get_text(strip=True)
    )

    return text.strip()

##antara scrapper

In [33]:
def scrape_antara(html):

    soup = BeautifulSoup(html, "html.parser")

    body = soup.find("div", class_="post-content")

    if body is None:
        body = soup.find("div", class_="post-content clearfix")

    if body is None:
        return None

    paragraphs = body.find_all("p")

    text = " ".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
        if p.get_text(strip=True)
    )

    return text.strip()

##universal scrapper

In [34]:
def scrape_article(row):

    html = download_page(row["url"])

    if html is None:
        return None

    if row["media"] == "Detik":

        return scrape_detik(html)

    elif row["media"] == "Antara":

        return scrape_antara(html)

    return None

##test

In [35]:
sample = all_links.iloc[0]

text = scrape_article(sample)

print(text[:1000])

Enzo Maresca resmi menjadi penerus Pep Guardiola di Manchester City . Dia rupanya sudah mengincar kursi pelatih The Sky Blues sejak masih membesut Chelsea . Man City mengumumkan penunjukan Maresca sebagai manajer baru tim pada Senin (29/6/2026) malam WIB. Juru taktik asal Italia itu diikat kontrak selama 3 tahun di Etihad Stadium. Maresca menjadi suksesor Guardiola yang telah menangani Man City selama 10 tahun. Sebelum menerima tawaran The Citizens , Maresca sempat melatih Chelsea pada 2024 hingga 2026. SCROLL TO CONTINUE WITH CONTENT Di Chelsea Maresca sukses mempersembahkan titel Piala Dunia Antarklub 2025. Keputusan mengejutkan diambilnya pada awal tahun 2026 dengan mengundurkan diri dari jabatan manajer The Blues. Beberapa laporan menyebut Maresca cabut dari Chelsea karena berselisih dengan direksi klub. London Biru justru berdalih pelatih berusia 36 tahun itu meninggalkan Stamford Bridge karena sudah yakin bakal jadi penerus Guardiola di Man City. "Pada musim gugur tahun lalu, Klu

#scrapping seluruh artikel

In [36]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import time
import random

def process_row(row):

    text = scrape_article(row)

    if text is None or len(text.strip()) < 100:
        return None

    return {
        "text": text,
        "media": row["media"],
        "label": row["label"]
    }


dataset = []

rows = [row for _, row in all_links.iterrows()]

with ThreadPoolExecutor(max_workers=8) as executor:

    results = executor.map(process_row, rows)

    for item in tqdm(results, total=len(rows)):

        if item is not None:
            dataset.append(item)

        time.sleep(random.uniform(0.1,0.2))

100%|██████████| 1104/1104 [03:26<00:00,  5.35it/s]


##buat dataframe

In [48]:
dataset_df = pd.DataFrame(dataset)

print(dataset_df.shape)

dataset_df.head()

(1104, 3)


,text,media,label
0,Enzo Maresca resmi menjadi penerus Pep Guardio...,Detik,Liga Inggris
1,Kepergian Enzo Maresca membuat Chelsea kalang ...,Detik,Liga Inggris
2,Enzo Maresca telah resmi menjadi manajer Manch...,Detik,Liga Inggris
3,Manchester City resmi mengumumkan Enzo Maresca...,Detik,Liga Inggris
4,Juergen Klopp enggan menghakimi musim Liverpoo...,Detik,Liga Inggris


In [49]:
dataset_df.groupby(["media","label"]).size()

media   label         
Antara  Liga Indonesia     79
        Liga Inggris      100
        Liga Italia        74
        Liga Spanyol       69
        Non Sepak Bola    160
Detik   Liga Indonesia    100
        Liga Inggris      100
        Liga Italia       100
        Liga Spanyol      100
        Non Sepak Bola    222
dtype: int64

##save and download dataset

In [50]:
import os

os.makedirs("dataset", exist_ok=True)

dataset_df.to_csv(
    "dataset/dataset1.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ dataset_final.csv berhasil disimpan!")
print(f"Total artikel : {len(dataset_df)}")

✅ dataset_final.csv berhasil disimpan!
Total artikel : 1104


In [51]:
from google.colab import files

files.download("dataset/dataset1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#B. Text Preprocessing

##import library

In [37]:
!pip install -q Sastrawi nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.5 MB/s eta 0:00:00


In [38]:
import re
import string

import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

##stopword and stemmer

In [39]:
stop_words = set(stopwords.words("indonesian"))

factory = StemmerFactory()

stemmer = factory.create_stemmer()

##cleansing

In [40]:
def cleansing(text):

    text = text.lower()

    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"www\S+", " ", text)

    text = re.sub(r"<.*?>", " ", text)

    text = re.sub(r"\n", " ", text)

    text = re.sub(r"\d+", " ", text)

    text = re.sub(r"[^\w\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

##tokenization

In [41]:
def tokenization(text):

    return word_tokenize(text)

##stopword removal

In [42]:
def filtering(tokens):

    result = []

    for word in tokens:

        if word not in stop_words:

            result.append(word)

    return result

##stemming

In [43]:
def stemming(tokens):

    hasil = []

    for token in tokens:

        hasil.append(
            stemmer.stem(token)
        )

    return hasil

##pipeline processing

In [44]:
def preprocess(text):

    text = cleansing(text)

    tokens = tokenization(text)

    tokens = filtering(tokens)

    tokens = stemming(tokens)

    return " ".join(tokens)

In [53]:
from tqdm import tqdm

tqdm.pandas()

dataset_df["clean_text"] = dataset_df["text"].progress_apply(preprocess)

100%|██████████| 1104/1104 [21:16<00:00,  1.16s/it]
